In [ ]:
# Configuração de importações e helpers
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

import main as q_main
import main_td
from helpers.data_loader import load_train_data, load_test_data
from ambiente.portfolio_env import PortfolioEnv
from agentes.Q_learning import AgentQLearning
from agentes.TD_learning import AgentTD

RESULTS_DIR = ROOT / "resultados"


def load_json(path):
    return json.loads(Path(path).read_text())


def load_saved_results():
    q_metrics = load_json(RESULTS_DIR / "metrics_qlearning.json")
    td_metrics = load_json(RESULTS_DIR / "metrics_td.json")
    q_history = load_json(RESULTS_DIR / "training_history.json")
    td_history = load_json(RESULTS_DIR / "training_history_td.json")
    q_eval = load_json(RESULTS_DIR / "eval_portfolio.json")
    td_eval = load_json(RESULTS_DIR / "eval_portfolio_td.json")
    return q_metrics, td_metrics, q_history, td_history, q_eval, td_eval


def show_metrics_table(q_metrics, td_metrics):
    df = pd.DataFrame([
        {
            "algorithm": "Q-Learning",
            "final_value": q_metrics["agent"]["final_value"],
            "total_return": q_metrics["agent"]["total_return"],
            "sharpe_ratio": q_metrics["agent"]["sharpe_ratio"],
            "max_drawdown": q_metrics["agent"]["max_drawdown"],
        },
        {
            "algorithm": "TD_LEARNING (TD)",
            "final_value": td_metrics["agent"]["final_value"],
            "total_return": td_metrics["agent"]["total_return"],
            "sharpe_ratio": td_metrics["agent"]["sharpe_ratio"],
            "max_drawdown": td_metrics["agent"]["max_drawdown"],
        },
    ])
    display(df.style.format({
        "final_value": "R$ {:,.2f}",
        "total_return": "{:.2%}",
        "sharpe_ratio": "{:.4f}",
        "max_drawdown": "{:.2%}",
    }))
    return df


def plot_training_history(q_history, td_history):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes[0, 0].plot(q_history["episode_rewards"], label="Q-Learning")
    axes[0, 0].plot(td_history["episode_rewards"], label="TD_LEARNING")
    axes[0, 0].set_title("Reward por episódio")
    axes[0, 0].set_xlabel("Episódio")
    axes[0, 0].set_ylabel("Reward")
    axes[0, 0].legend()

    axes[0, 1].plot(q_history["episode_portfolio_values"], label="Q-Learning")
    axes[0, 1].plot(td_history["episode_portfolio_values"], label="TD_LEARNING")
    axes[0, 1].set_title("Valor final do portfólio por episódio")
    axes[0, 1].set_xlabel("Episódio")
    axes[0, 1].set_ylabel("Valor final (R$)")
    axes[0, 1].legend()

    axes[1, 0].plot(q_history["episode_epsilons"], label="Q-Learning")
    axes[1, 0].plot(td_history["episode_epsilons"], label="TD_LEARNING")
    axes[1, 0].set_title("Epsilon ao final de cada episódio")
    axes[1, 0].set_xlabel("Episódio")
    axes[1, 0].set_ylabel("ε")
    axes[1, 0].legend()

    axes[1, 1].plot(q_history["episode_alphas"], label="Q-Learning")
    axes[1, 1].plot(td_history["episode_alphas"], label="TD_LEARNING")
    axes[1, 1].set_title("Alpha ao final de cada episódio")
    axes[1, 1].set_xlabel("Episódio")
    axes[1, 1].set_ylabel("α")
    axes[1, 1].legend()

    plt.tight_layout()
    plt.show()


def plot_eval_portfolios(q_eval, td_eval):
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(q_eval["portfolio_values"], label="Q-Learning")
    ax.plot(td_eval["portfolio_values"], label="TD_LEARNING (TD)")
    ax.set_title("Valor do portfólio no conjunto de teste")
    ax.set_xlabel("Passos de teste")
    ax.set_ylabel("Valor do portfólio (R$)")
    ax.legend()
    ax.grid(True)
    plt.show()


def run_learning_rate_experiment(agent_cls, label, alpha_values, n_episodes=200):
    train_df = load_train_data()
    test_df = load_test_data()
    env_config = q_main.ENV_CONFIG
    agent_config = q_main.AGENT_CONFIG.copy() if agent_cls is AgentQLearning else main_td.AGENT_CONFIG.copy()

    records = []
    for alpha in alpha_values:
        config = agent_config.copy()
        config["alpha"] = alpha
        config["alpha_min"] = max(alpha * 0.1, 0.001)
        config["alpha_decay"] = 0.995
        config["epsilon"] = 1.0
        config["epsilon_min"] = 0.01
        config["epsilon_decay"] = 0.995

        print(f"\n=== {label}: alpha={alpha} | episódios={n_episodes} ===")
        env_for_init = PortfolioEnv(train_df, **env_config)
        agent = agent_cls(env_for_init, **config)
        history = agent.train(PortfolioEnv(train_df, **env_config), n_episodes=n_episodes, log_interval=max(1, n_episodes // 4))
        eval_results = agent.evaluate(PortfolioEnv(test_df, **env_config))

        records.append({
            "algorithm": label,
            "alpha": alpha,
            "final_value": eval_results["final_value"],
            "total_return": eval_results["total_return"],
            "sharpe_ratio": eval_results["sharpe_ratio"],
            "max_drawdown": eval_results["max_drawdown"],
            "avg_reward_last_50": np.mean(history["episode_rewards"][-50:]),
            "n_states_visited": history["n_states_visited"],
        })

    return pd.DataFrame(records)


def plot_alpha_experiment(df):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    for name, group in df.groupby("algorithm"):
        axes[0, 0].plot(group["alpha"], group["total_return"], marker="o", label=name)
        axes[0, 1].plot(group["alpha"], group["sharpe_ratio"], marker="o", label=name)
        axes[1, 0].plot(group["alpha"], group["final_value"], marker="o", label=name)
        axes[1, 1].plot(group["alpha"], group["max_drawdown"], marker="o", label=name)

    axes[0, 0].set_title("Retorno total vs α")
    axes[0, 0].set_xlabel("α")
    axes[0, 0].set_ylabel("Retorno total")
    axes[0, 0].legend()
    axes[0, 0].grid(True)

    axes[0, 1].set_title("Sharpe vs α")
    axes[0, 1].set_xlabel("α")
    axes[0, 1].set_ylabel("Sharpe Ratio")
    axes[0, 1].legend()
    axes[0, 1].grid(True)

    axes[1, 0].set_title("Valor final do portfólio vs α")
    axes[1, 0].set_xlabel("α")
    axes[1, 0].set_ylabel("Valor final (R$)")
    axes[1, 0].legend()
    axes[1, 0].grid(True)

    axes[1, 1].set_title("Max Drawdown vs α")
    axes[1, 1].set_xlabel("α")
    axes[1, 1].set_ylabel("Max Drawdown")
    axes[1, 1].legend()
    axes[1, 1].grid(True)

    plt.tight_layout()
    plt.show()


In [1]:
# 2. Carregar resultados salvos e comparar desempenho
q_metrics, td_metrics, q_history, td_history, q_eval, td_eval = load_saved_results()

print("Métricas consolidadas dos agentes")
metrics_df = show_metrics_table(q_metrics, td_metrics)

print("Plot de métricas de treinamento")
plot_training_history(q_history, td_history)

print("Trajetória de valor do portfólio no teste")
plot_eval_portfolios(q_eval, td_eval)


## Conclusões e próximos passos

- Agora você tem um notebook que:
  - treina `main.py` (Q-Learning) e `main_td.py` (SARSA);
  - compara métricas de desempenho e curvas de treinamento;
  - explora a sensibilidade à taxa de aprendizado (`alpha`).

- Para aprofundar a análise, experimente também variar:
  - `epsilon_decay` e `alpha_decay`;
  - `transaction_cost` e `weight_delta` no ambiente;
  - número de bins (`n_bins`) para discretização.

- Os resultados de comparação ficam salvos em `resultados/` e podem ser reutilizados em novos notebooks ou relatórios.
